***DATA RECORD (Registro de Datos)***

<p align="center">
    <img src="../../assets/img/dataRecord.png" alt="Texto alternativo" width="450"/>
</p>

In [1]:
from typing import List, Dict
import numpy as np

In [2]:

# Parámetros que deberías obtener del header
data_path = "../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf"

n_signals: int = 30
samples_per_record: List[int] = [250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 1, 1, 1]
header_length: int = 256 + 256 * n_signals  # en bytes
total_samples_per_record: int = sum(samples_per_record)
print("total_samples_per_record:", total_samples_per_record)
bytes_per_sample: int = 2  # EDF típico usa int16
record_size_bytes: int = total_samples_per_record * bytes_per_sample

with open(data_path, "rb") as f:
    f.seek(header_length)  # saltamos el header completo

    # Leer el primer registro completo
    record_bytes = f.read(record_size_bytes)

    # Convertir a array de enteros
    record_values = np.frombuffer(record_bytes, dtype=np.int16)
    print("record_values.shape:", record_values.shape)

# Separar por canal según samples_per_record
canal_datos: List[np.ndarray] = []
start = 0
for nsamp in samples_per_record:
    end = start + nsamp
    canal_datos.append(record_values[start:end])
    start = end

# Mostrar algunas muestras del canal 0
print("Primeras muestras del canal 1:", canal_datos[0][:10])
print("lenght de canales:")
for i, canal in enumerate(canal_datos):
    print(f"Canal {i + 1}: {len(canal)} muestras")


total_samples_per_record: 6753
record_values.shape: (6753,)
Primeras muestras del canal 1: [-351 -418 -467 -435 -421 -414 -396 -316 -325 -305]
lenght de canales:
Canal 1: 250 muestras
Canal 2: 250 muestras
Canal 3: 250 muestras
Canal 4: 250 muestras
Canal 5: 250 muestras
Canal 6: 250 muestras
Canal 7: 250 muestras
Canal 8: 250 muestras
Canal 9: 250 muestras
Canal 10: 250 muestras
Canal 11: 250 muestras
Canal 12: 250 muestras
Canal 13: 250 muestras
Canal 14: 250 muestras
Canal 15: 250 muestras
Canal 16: 250 muestras
Canal 17: 250 muestras
Canal 18: 250 muestras
Canal 19: 250 muestras
Canal 20: 250 muestras
Canal 21: 250 muestras
Canal 22: 250 muestras
Canal 23: 250 muestras
Canal 24: 250 muestras
Canal 25: 250 muestras
Canal 26: 250 muestras
Canal 27: 250 muestras
Canal 28: 1 muestras
Canal 29: 1 muestras
Canal 30: 1 muestras


***1.- Exportar a un .csv la data digital de los canales 1-27***

***2.- Exportar a un .csv la data digital de los canales 28-30***

In [4]:
import numpy as np
import pandas as pd
from typing import List

# ============================
# Parámetros conocidos del archivo
# ============================
data_path = "../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf"

n_signals: int = 30
samples_per_record: List[int] = [250] * 27 + [1, 1, 1]
number_of_data_records: int = 1245
duration_of_data_record: float = 1.0  # segundos
header_length: int = 256 + 256 * n_signals
total_samples_per_record: int = sum(samples_per_record)
bytes_per_sample: int = 2
record_size_bytes: int = total_samples_per_record * bytes_per_sample

# ============================
# Leer todos los registros de datos
# ============================
with open(data_path, "rb") as f:
    f.seek(header_length)
    raw_data = f.read(record_size_bytes * number_of_data_records)

record_values = np.frombuffer(raw_data, dtype=np.int16)

# ============================
# Separar todos los datos por canal
# ============================
canal_datos = [
    np.zeros(samples_per_record[i] * number_of_data_records, dtype=np.int16)
    for i in range(n_signals)
]

offset = 0
for r in range(number_of_data_records):
    for c in range(n_signals):
        ns = samples_per_record[c]
        start = r * ns
        end = (r + 1) * ns
        canal_datos[c][start:end] = record_values[offset:offset + ns]
        offset += ns

# ============================
# Crear DataFrames separados
# ============================

# EEG: canales 1 al 27
eeg_data = np.stack(canal_datos[:27], axis=1)
fs_eeg = samples_per_record[0] / duration_of_data_record  # 250 Hz
tiempos_eeg = np.arange(eeg_data.shape[0]) / fs_eeg
df_eeg = pd.DataFrame(eeg_data, columns=[f"canal_{i+1}" for i in range(27)])
df_eeg.insert(0, "tiempo_s", tiempos_eeg)

# Canales lentos: 28 al 30
lento_data = np.stack(canal_datos[27:], axis=1)
tiempos_lento = np.arange(number_of_data_records)
df_lento = pd.DataFrame(lento_data, columns=[f"canal_{i+1}" for i in range(27, 30)])
df_lento.insert(0, "tiempo_s", tiempos_lento)

# Guardar ambos DataFrames en archivos CSV
df_eeg.to_csv("datos_EEG_exportado.csv", index=False)
df_lento.to_csv("datos_lentos_exportado.csv", index=False)


convertir a datos fisicos

In [4]:
# =======================================
# Exportar valores físicos de canales 1 al 27 desde archivo EDF
# =======================================

import numpy as np
import pandas as pd
from typing import List

# ============================
# Parámetros del archivo
# ============================
n_signals: int = 30
samples_per_record: List[int] = [250] * 27 + [1, 1, 1]
number_of_data_records: int = 1245
duration_of_data_record: float = 1.0
header_length: int = 256 + 256 * n_signals
bytes_per_sample: int = 2
total_samples_per_record: int = sum(samples_per_record)
record_size_bytes: int = total_samples_per_record * bytes_per_sample
data_path = "../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf"

# ============================
# Leer parámetros de conversión física
# ============================
digital_min, digital_max = [], []
physical_min, physical_max = [], []

with open(data_path, "rb") as f:
    # digital_min: offset = 256 + 120 * ns
    f.seek(256 + 120 * n_signals)
    dmin_bytes = f.read(n_signals * 8)
    digital_min = [int(dmin_bytes[i*8:(i+1)*8].decode().strip()) for i in range(n_signals)]

    # digital_max: offset = 256 + 128 * ns
    f.seek(256 + 128 * n_signals)
    dmax_bytes = f.read(n_signals * 8)
    digital_max = [int(dmax_bytes[i*8:(i+1)*8].decode().strip()) for i in range(n_signals)]

    # physical_min: offset = 256 + 104 * ns
    f.seek(256 + 104 * n_signals)
    pmin_bytes = f.read(n_signals * 8)
    physical_min = [float(pmin_bytes[i*8:(i+1)*8].decode().strip()) for i in range(n_signals)]

    # physical_max: offset = 256 + 112 * ns
    f.seek(256 + 112 * n_signals)
    pmax_bytes = f.read(n_signals * 8)
    physical_max = [float(pmax_bytes[i*8:(i+1)*8].decode().strip()) for i in range(n_signals)]

# ============================
# Leer datos binarios
# ============================
with open(data_path, "rb") as f:
    f.seek(header_length)
    raw_data = f.read(record_size_bytes * number_of_data_records)

record_values = np.frombuffer(raw_data, dtype=np.int16)

# ============================
# Separar datos por canal
# ============================
canal_datos_digitales = [
    np.zeros(samples_per_record[i] * number_of_data_records, dtype=np.int16)
    for i in range(n_signals)
]

offset = 0
for r in range(number_of_data_records):
    for c in range(n_signals):
        ns = samples_per_record[c]
        start = r * ns
        end = (r + 1) * ns
        canal_datos_digitales[c][start:end] = record_values[offset:offset + ns]
        offset += ns

# ============================
# Convertir canales 1 al 27 a valores físicos
# ============================
canal_datos_fisicos = []
for i in range(27):
    d_vals = canal_datos_digitales[i]
    dmin = digital_min[i]
    dmax = digital_max[i]
    pmin = physical_min[i]
    pmax = physical_max[i]
    f_vals = ((d_vals - dmin) / (dmax - dmin)) * (pmax - pmin) + pmin
    canal_datos_fisicos.append(f_vals)

# Construir DataFrame y guardar CSV
eeg_fisico_data = np.stack(canal_datos_fisicos, axis=1)
fs_eeg = samples_per_record[0] / duration_of_data_record  # 250 Hz
tiempos_eeg = np.arange(eeg_fisico_data.shape[0]) / fs_eeg
df_eeg_fisico = pd.DataFrame(eeg_fisico_data, columns=[f"canal_{i+1}" for i in range(27)])
df_eeg_fisico.insert(0, "tiempo_s", tiempos_eeg)

df_eeg_fisico.to_csv("datos_EEG_fisico_exportado.csv", index=False)


***Código sugerido para guardar con MNE***

In [6]:
import mne
import pandas as pd
import numpy as np

# Cargar archivo EDF
raw = mne.io.read_raw_edf(data_path, preload=True, verbose=False)

# Extraer datos físicos y tiempos
data, times = raw[:27, :]  # canales 1 al 27

# Transponer para tener muestras en filas
data = data.T

# Crear DataFrame
df_mne = pd.DataFrame(data, columns=[f"canal_{i+1}" for i in range(27)])
df_mne.insert(0, "tiempo_s", times)

# Guardar a CSV
df_mne.to_csv("datos_EEG_fisico_mne.csv", index=False)
